# Face model v2 — round 7: fine-tune HSEmotion EfficientNet-B2 (AffectNet) on FER+
Off-the-shelf B2 (AffectNet) beats our FER2013 models on labelled video (RAVDESS) but over-reacts on signers. This round starts from its weights
(ONNX -> PyTorch with onnx2torch) and trains on FER+ soft labels, with a small LR and distillation to the original B2 to keep its real-face skill.
Epoch 0 in each log = the original B2. Checkpoint = FER+ val F1 + 0.5 x RAVDESS actors 1-2 clip F1; actors 3-4 are tested on the laptop.

Needs in `MyDrive/face-v2/`: `Face-Lab-code-r7.zip`, the FER2013 zip. Runtime → T4 GPU.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)
else:
    raise SystemExit('NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, zipfile
D='/content/drive/MyDrive/face-v2'
os.makedirs('/content/Main_demo', exist_ok=True)
zipfile.ZipFile(f'{D}/Face-Lab-code-r7.zip').extractall('/content/Main_demo')
fer = [p for p in glob.glob(f'{D}/*.zip') if 'Face-Lab' not in p][0]
zipfile.ZipFile(fer).extractall('/content/Main_demo/FER2013-Data')
for s in ('train','test'):
    print(s, sum(len(f) for _,_,f in os.walk(f'/content/Main_demo/FER2013-Data/{s}')))   # expect 28709 / 7178
%cd /content/Main_demo/Face-Lab
print(sorted(os.listdir('results')))
!pip -q install onnx onnx2torch onnxruntime
!mkdir -p hse && wget -q -O hse/enet_b2_8.onnx https://github.com/av-savchenko/face-emotion-recognition/raw/main/models/affectnet_emotions/onnx/enet_b2_8.onnx && ls -la hse

In [ ]:
EXPS = {
  'b2_head':     '--head-only --epochs 3 --lr 1e-4 --kd 0',
  'b2_ft_kd1':   '--epochs 3 --lr 2e-5 --kd 1.0',
  'b2_ft_kd0':   '--epochs 3 --lr 2e-5 --kd 0',
}
for tag, extra in EXPS.items():
    cmd = f'python ft_b2.py {extra} --tag {tag}'
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    lines = r.stdout.strip().splitlines()
    print(chr(10).join(lines[-6:]) if lines else '', r.stderr[-1500:] if r.returncode else '', flush=True)


In [ ]:
import json, glob
for f in sorted(glob.glob('results/b2_*.json')):
    r = json.load(open(f))
    print(r['tag'], 'best epoch', r['best_epoch'])
    for row in r['log']:
        print('   ep', row['epoch'], 'FER+ F1 %.4f | RAV-dev F1 %.3f exprRec %.3f neuKept %.3f' % (row['val_fp_macro_f1'], row['rav_dev_macro_f1'], row['rav_dev_expr_recall'], row['rav_dev_neutral_kept']))

In [ ]:
OUT='/content/drive/MyDrive/face-v2/out'
shutil.make_archive('/content/face_v2_results_r7','zip','/content/Main_demo/Face-Lab','results')
shutil.copy('/content/face_v2_results_r7.zip', OUT)
for t in EXPS:
    if os.path.exists(f'runs/{t}/best.onnx'):
        os.makedirs(f'{OUT}/{t}', exist_ok=True)
        b = open(f'runs/{t}/best.onnx','rb').read()
        for i in range(0, len(b), 9000000):                     # parts < 10 MB so they can be pulled back to the laptop
            open(f'{OUT}/{t}/best.onnx.part{i // 9000000}','wb').write(b[i:i + 9000000])
print(sorted(os.listdir(OUT)))